In [66]:
import csv
import glob
import os

def create_csv(file_name):
    #create csv with heading
    with open(file_name, 'w', newline='',encoding='utf-8') as csvfile:
        writer = csv.writer(csvfile)
        #writer.writerow(['nr', 'bid','brewery','name','kind', 'link','abv','ibu', 'rating','nr_raters','adding_date','text'])

def beer_to_csv(data,file_name):
    #add new entry into csv
    with open(file_name, 'a', newline='',encoding='utf-8') as csvfile:
        writer = csv.writer(csvfile)
        writer.writerows(data)

def Open_Csv(name):
    with open(name, "r+", encoding="utf-8") as file:
        reader = csv.reader(file)
        data=[]
        for row in reader:
            data.append(row)
    return data

def Look_Eliminate_Duplicates(in_data):
    seen = set()
    filtered_data = []

    for row in in_data:
        value = row[1]
        if value not in seen:
            seen.add(value)
            filtered_data.append(row)
    
    return filtered_data

def Keyword_Matching(in_beer_data):
    for row in in_beer_data:
        if ("malt" in row[11]) or ("maláta" in row[11]):
            row.append(1) #row no 12
        else:
            row.append(0)
        if ("hops" in row[11]) or ("komló" in row[11]):
            row.append(1) #row no 13
        else:
            row.append(0)
        if ("chocolate" in row[11]) or ("csokoládé" in row[11])or ("csoki" in row[11]):
            row.append(1) #row no 14
        else:
            row.append(0)
        if ("German" in row[11]) or ("német" in row[11])or ("german" in row[11]):
            row.append(1) #row no 15
        else:
            row.append(0)
        if ("banana" in row[11]) or ("banán" in row[11]):
            row.append(1) #row no 16
        else:
            row.append(0)
        if ("vanilla" in row[11]) or ("vanília" in row[11]):
            row.append(1) #row no 17
        else:
            row.append(0)
        if ("caramel" in row[11]) or ("karamella" in row[11]):
            row.append(1) #row no 18
        else:
            row.append(0)
        if ("yeast" in row[11]) or ("élesztő" in row[11]):
            row.append(1) #row no 19
        else:
            row.append(0)
        if ("fruit" in row[11]) or ("gyümölcs" in row[11]) or ("berry" in row[11])or ("berries" in row[11]):
            row.append(1) #row no 20
        else:
            row.append(0)
        if ("bourbon" in row[11]) or ("whiskey" in row[11]):
            row.append(1) #row no 21
        else:
            row.append(0)
        if ("rhum" in row[11]) or ("rum" in row[11]):
            row.append(1) #row no 22
        else:
            row.append(0)
        if ("alcohol" in row[11]) or ("alkohol" in row[11]):
            row.append(1) #row no 23
        else:
            row.append(0)
        if ("gluten" in row[11]) or ("glutén" in row[11]):
            row.append(1) #row no 24
        else:
            row.append(0)
        if ("barrel" in row[11]) or ("hordó" in row[11]):
            row.append(1) #row no 25
        else:
            row.append(0)
        if ("rye" in row[11]) or ("rozs" in row[11]):
            row.append(1) #row no 26
        else:
            row.append(0)
        if ("barley" in row[11]) or ("árpa" in row[11]):
            row.append(1) #row no 27
        else:
            row.append(0)
        if ("cocoa" in row[11]) or ("kakaó" in row[11]):
            row.append(1) #row no 28
        else:
            row.append(0)
        if ("alacsony" in row[11]) or ("alsó" in row[11]) or ("kicsi" in row[11]) or ("kevés" in row[11]) or ("pici" in row[11]) or ("hideg" in row[11]):
            row.append(1) #row no 29
        else:
            row.append(0)
        if ("magas" in row[11]) or ("felső" in row[11]) or ("nagy" in row[11]) or ("sok" in row[11]) or ("meleg" in row[11]):
            row.append(1) #row no 30
        else:
            row.append(0)
    return in_beer_data

#QA
#1 remove duplications from both csv-s
#remove duplicates from Top50
Top50Data=Open_Csv("../Scraper/Top50Data.csv")
Top50Data = Look_Eliminate_Duplicates(Top50Data)
#remove duplicates from AllData
AllData=Open_Csv("../Scraper/AllData.csv")
AllData = Look_Eliminate_Duplicates(AllData)
#remove Top50Data from AllData based on row[1]
top_keys = {row[1] for row in Top50Data}
AllData = [row for row in AllData if row[1] not in top_keys]

#2 set up 3 csv-s based on the 2 with cutoff / treshold value
treshold=3.85
good_beers = []
rest_beers = []
for row in AllData:
    try:
        value = float(row[8])
        if value > treshold:
            good_beers.append(row)
        else:
            rest_beers.append(row)

    except (ValueError, IndexError):
        pass
        
#3 adding one hot encoding variables and other new variables
datas=[Top50Data,good_beers,rest_beers]
for data_type in datas:
    data_type = Keyword_Matching(data_type)

#4 export Csvs
#remove all the CSVs from the folder
csv_files = glob.glob("*.csv")
for file in csv_files:
   os.remove(file)
   print(f"Törölve: {file}")
    
files=["TopBeerData.csv","GoodBeerData.csv","RestBeerData.csv"]
for i in range (0,len(files)):
    create_csv(files[i])
    beer_to_csv(datas[i],files[i])
    print(f"{files[i]} file created")

Törölve: TopBeerData.csv
TopBeerData.csv file created
GoodBeerData.csv file created
RestBeerData.csv file created


In [92]:
#DV: rating row[8]
#IDVs: abv - row[6], raters - row[9], text - row[12]-row[30]
from sklearn.linear_model import LinearRegression
import numpy as np

datas=[Top50Data,good_beers,rest_beers]
for data in datas:
    #data = data[1:]
    numeric_data = []
    for row in data[1:]: #ignore heading
        numeric_data.append([
            0 if row[6] == "N/A" else float(row[6]),
            float(row[9]),
            float(row[12]),
            float(row[13]),
            float(row[14]),
            float(row[15]),
            float(row[16]),
            float(row[17]),
            float(row[18]),
            float(row[19]),
            float(row[20]),
            float(row[21]),
            float(row[22]),
            float(row[23]),
            float(row[24]),
            float(row[25]),
            float(row[26]),
            float(row[27]),
            float(row[28]),
            float(row[29]),
            float(row[30]),
            float(row[8])
        ])
    numeric_data = np.array(numeric_data)
    #print(numeric_data)
    X = numeric_data[:, :-1]
    y = numeric_data[:, -1]
    model = LinearRegression()
    model.fit(X, y)
    print("coefs",model.coef_)
    print("intercept",model.intercept_)
    #for i in range(X.shape[1]):
     #   print(np.corrcoef(X[:, i], y)[0, 1])

coefs [-2.65918067e-04  1.97607014e-05 -7.47041820e-03  8.51578933e-04
 -1.23541505e-02 -1.20871688e-02 -1.38777878e-17  6.02674573e-02
  1.82256344e-01  5.85109014e-02 -2.12191976e-02 -2.53800580e-02
  3.29359411e-02 -1.20871688e-02 -5.20417043e-18  1.68075498e-03
 -1.21127428e-01  0.00000000e+00  3.14981772e-02  1.33088211e-02
 -4.72339460e-02]
intercept 4.096385342222417
0.02134814998709061
0.37873064022149283
0.019640531005095024
-0.1839244660175601
0.10051330573195648
-0.11167452623483666
nan
0.2294555318991159
0.20866619986247378
0.0040914941149506655
-0.05427980324495132
0.1306926159690291
0.020044145734459107
-0.11167452623483666
nan
0.035305952561647994
0.002863449390636948
nan
0.07979439752664223
-0.04557658275830976
-0.12247175342517322
coefs [ 6.29651651e-03  1.86048813e-05 -6.27677285e-03  4.08474217e-02
  1.97154092e-02  3.97223316e-02  5.36669631e-02  2.35559127e-02
 -1.91169102e-02 -1.01246581e-02 -2.80792989e-03  3.68137059e-03
  1.66826606e-02 -5.24650570e-02  0.00000

C:\Users\User\AppData\Local\Programs\Python\Python314\Lib\site-packages\numpy\lib\_function_base_impl.py:3023: RuntimeWarning: invalid value encountered in divide
  c /= stddev[:, None]
C:\Users\User\AppData\Local\Programs\Python\Python314\Lib\site-packages\numpy\lib\_function_base_impl.py:3024: RuntimeWarning: invalid value encountered in divide
  c /= stddev[None, :]
C:\Users\User\AppData\Local\Programs\Python\Python314\Lib\site-packages\numpy\lib\_function_base_impl.py:3023: RuntimeWarning: invalid value encountered in divide
  c /= stddev[:, None]
C:\Users\User\AppData\Local\Programs\Python\Python314\Lib\site-packages\numpy\lib\_function_base_impl.py:3024: RuntimeWarning: invalid value encountered in divide
  c /= stddev[None, :]


In [90]:
import numpy as np
import matplotlib.pyplot as plt

row = data[0]
print(data[0])

x = np.arange(len(row))

y = np.array([
    0 if v == "N/A" else float(v)
    for v in row
])

m, b = np.polyfit(x, y, 1)

plt.scatter(x, y)
plt.plot(x, m*x + b)
plt.show()

['0', '1059193', 'Thomas Menner 1701', 'Félbarna, Altbier', 'Altbier - Sticke / Latzenbier', 'https://untappd.com/b/thomas-menner-1701-felbarna-altbier/1059193', '6.2', 'N/A', '3.59', '235', '04/24/15', ' ', 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]


ValueError: could not convert string to float: 'Thomas Menner 1701'